Describe & load the data

In [2]:
import pandas as pd
df = pd.read_csv("C:\\Users\\Asus\\Desktop\\data for project coach you\\ai4i2020.csv")
print(df.head())

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    0    0    0  
2    0  

In [4]:
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

(10000, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(

install sdv

In [6]:
pip install sdv

   ---------------------------------------- 0.0/15.9 MB ? eta -:--:--
    --------------------------------------- 0.3/15.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/15.9 MB 2.2 MB/s eta 0:00:08
   -- ------------------------------------- 1.0/15.9 MB 1.7 MB/s eta 0:00:09
   --- ------------------------------------ 1.3/15.9 MB 1.8 MB/s eta 0:00:09
   ---- ----------------------------------- 1.8/15.9 MB 1.9 MB/s eta 0:00:08
   ----- ---------------------------------- 2.4/15.9 MB 2.1 MB/s eta 0:00:07
   ------- -------------------------------- 2.9/15.9 MB 2.1 MB/s eta 0:00:07
   -------- ------------------------------- 3.4/15.9 MB 2.2 MB/s eta 0:00:06
   --------- ------------------------------ 3.9/15.9 MB 2.2 MB/s eta 0:00:06
   ---------- ----------------------------- 4.2/15.9 MB 2.2 MB/s eta 0:00:06
   ------------ --------------------------- 5.0/15.9 MB 2.2 MB/s eta 0:00:05
   -------------- ------------------------- 5.8/15.9 MB 2.4 MB/s eta 0:00:05
   ----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.19.0 requires botocore<1.36.4,>=1.36.0, but you have botocore 1.43.103 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


CTGANSynthesizer for synthetic data

In [13]:
import pandas as pd  # للتعامل مع الداتا

from sdv.metadata import Metadata  # لمعرفة أنواع الأعمدة
from sdv.single_table import CTGANSynthesizer  # موديل توليد البيانات


df = pd.read_csv("C:\\Users\\Asus\\Desktop\\data for project coach you\\ai4i2020.csv")  # قراءة الداتا الأصلية

failures = ["TWF", "HDF", "PWF", "OSF", "RNF"]  # أنواع الأعطال الخمسة

synthetic_all = []  # قائمة لتخزين كل البيانات الجديدة


for failure in failures:  # المرور على كل نوع عطل

    data = df[df[failure] == 1].copy()  # أخذ بيانات هذا العطل فقط

    """"
    data = data.drop(
        columns=["UDI", "Product ID"],  # حذف الأعمدة غير المهمة
        errors="ignore"  # تجاهل الخطأ إذا العمود غير موجود
    )
    """

    metadata = Metadata.detect_from_dataframe(
        data=data,  # إعطاء الداتا للموديل
        table_name="machine_failure"  # اسم الجدول
    )

    model = CTGANSynthesizer(
        metadata,  # معلومات الداتا
        epochs=100  # عدد مرات التدريب
    )

    model.fit(data)  # تدريب الموديل

    synthetic = model.sample(
        num_rows=1000  # توليد 1000 سجل جديد
    )

    synthetic["Machine failure"] = 1  # اعتبارها ماكينة فيها عطل

    synthetic[failure] = 1  # تحديد نوع العطل

    synthetic_all.append(synthetic)  # إضافة البيانات للقائمة


synthetic_all = pd.concat(synthetic_all, ignore_index=True)  # دمج كل الأعطال


final_data = pd.concat(
    [df, synthetic_all],  # دمج الداتا الأصلية مع الـ Synthetic
    ignore_index=True  # إعادة ترتيب أرقام الصفوف
)


final_data.to_csv(
    r"C:\Users\Asus\Desktop\AI4I_with_synthetic_data_1000.csv",  # اسم الملف النهائي
    index=False  # عدم حفظ أرقام الصفوف
)


print("Original data:", len(df))  # عدد الصفوف الأصلية

print("Synthetic data:", len(synthetic_all))  # عدد الصفوف الجديدة

print("Final data:", len(final_data))  # عدد الصفوف بعد الدمج

c:\Users\Asus\anaconda3\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Asus\anaconda3\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Asus\anaconda3\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Asus\anaconda3\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
c:\Users\Asus\anaconda3\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in fut

Original data: 10000
Synthetic data: 5000
Final data: 15000


Describe & load the synthetic data

In [11]:
import pandas as pd
df = pd.read_csv("C:\\Users\\Asus\\Desktop\\data for project coach you\\AI4I_with_synthetic_data.csv")
print(df.head())

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    0    0    0  
2    0  

In [12]:
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

(12500, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      12500 non-null  int64  
 1   Product ID               12500 non-null  object 
 2   Type                     12500 non-null  object 
 3   Air temperature [K]      12500 non-null  float64
 4   Process temperature [K]  12500 non-null  float64
 5   Rotational speed [rpm]   12500 non-null  int64  
 6   Torque [Nm]              12500 non-null  float64
 7   Tool wear [min]          12500 non-null  int64  
 8   Machine failure          12500 non-null  int64  
 9   TWF                      12500 non-null  int64  
 10  HDF                      12500 non-null  int64  
 11  PWF                      12500 non-null  int64  
 12  OSF                      12500 non-null  int64  
 13  RNF                      12500 non-null  int64  
dtypes: float64